In [1]:
import pandas as pd
import numpy as np
from scipy import stats

# Загрузка данных
df = pd.read_csv(r'C:\Users\nikka\OneDrive\Рабочий стол\ОТП Задания\Python Analytics\sms_ab_test.csv')

In [2]:
df.head()

,customer_id,group_name,segment,delivered_flg,paid_7d_flg,paid_amount_7d
0,300000,control,high_risk,1,0,0.0
1,300001,treatment,medium_risk,1,0,0.0
2,300002,control,medium_risk,1,0,0.0
3,300003,treatment,high_risk,1,0,0.0
4,300004,control,medium_risk,1,0,0.0


In [3]:
# Создадим функцию для расчета метрик
def metrics_calc(data):
    return pd.Series({'customers_cnt': len(data), 'delivery_rate': data['delivered_flg'].mean(), 
                      'payment_rate_7d': data['paid_7d_flg'].mean(), 'avg_paid_amount_per_customer_7d': data['paid_amount_7d'].mean()})

In [4]:
# Расчитаем метрики

# Расчитаем метрики по сегментам
segment_metrics = (df.groupby(['segment', 'group_name']).apply(metrics_calc).reset_index())

# Расчитаем метрики по группам
group_metrics = (df.groupby('group_name').apply(metrics_calc).reset_index())

print("Метрики по сегментам")
print(segment_metrics)

print("\nМетрики по группам")
print(group_metrics)

Метрики по сегментам
       segment group_name  customers_cnt  delivery_rate  payment_rate_7d  \
0    high_risk    control          863.0       0.900348         0.123986   
1    high_risk  treatment          854.0       0.879391         0.148712   
2     low_risk    control          987.0       0.959473         0.313070   
3     low_risk  treatment          987.0       0.954407         0.372847   
4  medium_risk    control         1296.0       0.932099         0.221451   
5  medium_risk  treatment         1324.0       0.939577         0.237915   
6    newcomers    control          454.0       0.916300         0.180617   
7    newcomers  treatment          435.0       0.935632         0.204598   

   avg_paid_amount_per_customer_7d  
0                      1234.474762  
1                      1424.542541  
2                      1763.592665  
3                      2083.512199  
4                      1623.122955  
5                      1767.628595  
6                      1191.874802 

C:\Users\nikka\AppData\Local\Temp\ipykernel_39328\1086585932.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  segment_metrics = (df.groupby(['segment', 'group_name']).apply(metrics_calc).reset_index())
C:\Users\nikka\AppData\Local\Temp\ipykernel_39328\1086585932.py:7: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  group_metrics = (df.groupby('group_name').apply(metrics_calc).reset_index())


In [5]:
# Посчитаем изменение

# Узнаем долю пользователей, совершивших оплату в течение 7 дней из контрольной группы
control_rate = group_metrics.loc[group_metrics['group_name'] == 'control', 'payment_rate_7d'].iloc[0]

# Узнаем долю пользователей, совершивших оплату в течение 7 дней из тестовой группы
treatment_rate = group_metrics.loc[group_metrics['group_name'] == 'treatment', 'payment_rate_7d'].iloc[0]

# Считаем изменение
uplift = (treatment_rate - control_rate) / control_rate

print(f"Измение конверсии в оплату в тестовой группе относительно контрольной группы: {uplift:.2%}")

Измение конверсии в оплату в тестовой группе относительно контрольной группы: 14.52%


In [6]:
# Проведем статистический тест

# Поделим данные на группы
control = df[df['group_name'] == 'control']
treatment = df[df['group_name'] == 'treatment']

# Посчитаем кол-во оплат для каждой группы
paid_control = control['paid_7d_flg'].sum()
paid_treatment = treatment['paid_7d_flg'].sum()

# Посчитаем размер выборок
n_control = len(control)
n_treatment = len(treatment)

# Расчитаем конверсии в оплату в обеих группах
p1 = paid_control / n_control
p2 = paid_treatment / n_treatment

# Расчитаем общую конверсии в оплату по обеим группам
p_pool = (paid_control + paid_treatment) / (n_control + n_treatment)

# Посчитаем стандартную ошибку
se = np.sqrt(p_pool * (1 - p_pool) * (1/n_control + 1/n_treatment))

# Используем z-статистику - так как мы сравниваем группы по бинарной метрике(0 - оплаты не было, 1 - оплата была) 
# и для того типа метрики z-статистика подходит лучше всего
z = (p2 - p1) / se

# Расчитаем p-value, используя двухсторонний тест
p_value = 2 * (1 - stats.norm.cdf(abs(z)))

print("Результаты статистического теста (z-test)")
print(f"z-stat: {z:.4f}")
print(f"p-value: {p_value:.4f}")



Результаты статистического теста (z-test)
z-stat: 3.1739
p-value: 0.0015


In [7]:
# Сформируем короткий вывод

# Используем статистическую значимость равную 5%
alpha = 0.05

# Оценим изменение конверсии
if uplift > 0:
    direction = "увеличивает"
elif uplift < 0:
    direction = "снижает"
else:
    direction = "не изменяет"

# Формируем вывод
if p_value < alpha:
    conclusion = (
        f"Новый вариант SMS {direction} конверсию в оплату: uplift составил {uplift:.2%}. "
        f"\nРазличие статистически значимо (p-value = {p_value:.4f} < {alpha}), "
        f"поэтому эффект с высокой вероятностью не является случайным. "
    )
else:
    conclusion = (
        f"Новый вариант SMS {direction} конверсию в оплату: uplift составил {uplift:.2%}. "
        f"\nОднако различие статистически незначимо (p-value = {p_value:.4f} ≥ {alpha}), "
        f"поэтому нельзя уверенно говорить о наличии эффекта. "
        f"\nРекомендуется продолжить тест или увеличить выборку."
    )

print("Вывод по тесту:")
print(conclusion)

Вывод по тесту:
Новый вариант SMS увеличивает конверсию в оплату: uplift составил 14.52%. 
Различие статистически значимо (p-value = 0.0015 < 0.05), поэтому эффект с высокой вероятностью не является случайным. 
